<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 2 · XGBoost & Random Forest</h4>
<h1 align="center">Random Forest — Hotel Booking Demand</h1>
<p align="center"><i>Loading the raw data and confronting the impurity it already ships with</i></p>

---


## 1. Why this notebook exists

Before we can grow a single decision tree — let alone a forest of them — we need to know
**what we are actually working with**. This notebook has one job: load the raw Hotel
Booking Demand dataset, understand its shape, its missing values, and a very specific
quality problem it has (duplicate rows), and produce a clean(er) file that the rest of the
Random Forest sessions will build on.

We are **not** doing feature engineering or leakage removal here — that is the job of
`02_data_processing.ipynb`. This notebook is deliberately scoped to *inspection* and
*deduplication* only, because those two things deserve to be understood on their own before
we start reshaping columns.

> **Where this fits in the pipeline**
> `00_dataset_and_impurity.ipynb` → `01_eda.ipynb` → `02_data_processing.ipynb` →
> `03_train_test_eval.ipynb`


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA_PATH = "data/hotel_bookings_full.csv"
df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 119,390  |  Columns: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03



## 2. First look: structure and types

`.info()` gives us dtypes and non-null counts in one shot — this is always the first command
to run on any new dataset. Pay attention to two things:

- Columns that *should* be numeric but are loaded as `float64` because they contain missing
  values (`children`, `agent`, `company` — pandas silently upcasts an integer column with
  `NaN`s to float, since integer arrays cannot represent `NaN`).
- Columns that are clearly categorical text (`hotel`, `meal`, `market_segment`,
  `deposit_type`, `customer_type`, `reservation_status`, ...) — these will need encoding
  before any tree-based model can use them numerically, which we handle in the processing
  notebook.


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [3]:
df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


In [4]:
df.describe(include="object").T

C:\Users\Asus\AppData\Local\Temp\ipykernel_26196\1274302342.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object").T


,count,unique,top,freq
hotel,119390,2,City Hotel,79330
arrival_date_month,119390,12,August,13877
meal,119390,5,BB,92310
country,118902,177,PRT,48590
market_segment,119390,8,Online TA,56477
distribution_channel,119390,5,TA/TO,97870
reserved_room_type,119390,10,A,85994
assigned_room_type,119390,12,A,74053
deposit_type,119390,3,No Deposit,104641
customer_type,119390,4,Transient,89613



## 3. Missing values

Random Forests (and CART-style trees generally) do not automatically handle missing values
in scikit-learn's implementation — every `NaN` has to be resolved one way or another before
we fit anything. Let's quantify exactly how much is missing and where.


In [5]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary

,missing_count,missing_pct
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00



Four columns carry missing values, and they are missing for very different reasons — this
matters because it will shape *how* we fill them later:

- **`company`** — missing in roughly **94%** of rows. This isn't really "missing" so much as
  "not applicable": the overwhelming majority of bookings are made by individuals, not
  companies. A column that is 94% empty is not usably imputable — any value we invent for it
  would be almost pure noise. (We drop this column entirely in the processing notebook.)
- **`agent`** — missing in roughly **13.7%** of rows. Similarly, a booking with no agent ID
  most likely means *no travel agent was involved*, not that the agent ID was lost. This is
  informative missingness, not random missingness.
- **`country`** — missing in **0.4%** of rows, a small number, most plausibly genuine data
  entry gaps.
- **`children`** — missing in only **4 rows** (a rounding error next to 119,390 rows) —
  almost certainly safe to fill with `0`, the mode/median for this column.

> **Key idea:** missingness is not one phenomenon. `company` and `agent` are missing
> *because a real-world event didn't happen* (no company was involved, no agent was used).
> `country` and `children` are missing more like ordinary data-entry gaps. Treating all four
> the same way (e.g. blanket mean-imputation) would be a mistake — we revisit each of them
> individually in `02_data_processing.ipynb`.


In [6]:
for col in ["children", "country", "agent", "company"]:
    n_missing = df[col].isna().sum()
    pct = n_missing / len(df) * 100
    print(f"{col:10s}: {n_missing:>7,} missing  ({pct:5.2f}%)  dtype={df[col].dtype}")

children  :       4 missing  ( 0.00%)  dtype=float64
country   :     488 missing  ( 0.41%)  dtype=str
agent     :  16,340 missing  (13.69%)  dtype=float64
company   : 112,593 missing  (94.31%)  dtype=float64



## 4. The "impurity" for this topic: duplicate rows

Every Random Forest topic in this workshop pairs a modeling concept with a data-quality
concept. For Random Forest, the paired concept is **duplicate records** — and here is the
important twist:

> **We do not need to inject this impurity. This dataset already ships with it.** That is
> itself the lesson: real-world data is frequently dirty *before you have ever touched it*.
> Nobody hands you a pristine CSV — you inherit whatever mess the source system produced, and
> the first job of any analyst or ML engineer is to go looking for that mess rather than
> assume it isn't there.

Let's check for fully-duplicated rows — rows where **all 32 columns** are identical.


In [7]:
n_duplicates = df.duplicated().sum()
pct_duplicates = n_duplicates / len(df) * 100
print(f"Fully duplicated rows: {n_duplicates:,} out of {len(df):,} ({pct_duplicates:.2f}%)")

Fully duplicated rows: 31,994 out of 119,390 (26.80%)



That is not a typo — roughly **1 in 4 rows** in this dataset is an exact duplicate of another
row, matching on every single one of the 32 columns, including things like `lead_time`,
`arrival_date_day_of_month`, `adr`, `agent`, and `reservation_status_date`.

### Are these really "duplicates"?

Here is the genuinely tricky part, and it's worth sitting with it rather than rushing past
it: **this dataset has no booking-ID column, or any other primary-key column at all.**

Check the column list below — nothing uniquely identifies a booking as *this specific
reservation* independent of its attributes.


In [8]:
print(list(df.columns))

['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']



Without a primary key, we genuinely cannot *prove* these are duplicate log entries for the
same booking. Two possibilities are both technically consistent with what we observe:

1. **Duplicate records** — the same booking was logged twice (e.g. a system retry, a batch
   re-import, an export that ran twice and got concatenated). This is the far more likely
   explanation in practice.
2. **Legitimately identical bookings** — two *different* guests who happen to match on
   hotel, arrival date, length of stay, number of adults/children/babies, meal plan, country,
   market segment, distribution channel, room type, deposit type, ADR, and every other field,
   right down to the reservation status and its date. This is statistically implausible for
   any single pair of columns to coincide on all 32 dimensions simultaneously — but with
   119,390 rows and many low-cardinality categorical columns, some collisions are not
   impossible either.

> **Why this matters beyond this dataset:** this is exactly why production systems assign a
> primary key (a booking ID, an order ID, a transaction ID) to every record the moment it is
> created. A primary key turns "are these the same event?" from a statistical guessing game
> into a simple equality check. Its absence here is a design flaw in how this data was
> published, and it is a completely realistic one — plenty of real systems you will work with
> professionally have the same gap.

Given the scale of the effect (nearly a quarter of the dataset) and the strong prior that
double-logging is more likely than genuine coincidence at this rate, we will proceed by
**dropping exact duplicates**. But we will not do this silently — we report the effect on the
headline number a stakeholder would actually care about: the overall cancellation rate.


In [9]:
cancel_rate_before = df["is_canceled"].mean()
df_clean = df.drop_duplicates().reset_index(drop=True)
cancel_rate_after = df_clean["is_canceled"].mean()

print(f"Rows before dedup: {len(df):,}")
print(f"Rows after  dedup: {len(df_clean):,}")
print(f"Rows removed:      {len(df) - len(df_clean):,}")
print()
print(f"Cancellation rate before dedup: {cancel_rate_before:.4%}")
print(f"Cancellation rate after  dedup: {cancel_rate_after:.4%}")
print(f"Absolute shift:                 {(cancel_rate_after - cancel_rate_before):.4%}")

Rows before dedup: 119,390
Rows after  dedup: 87,396
Rows removed:      31,994

Cancellation rate before dedup: 37.0416%
Cancellation rate after  dedup: 27.4898%
Absolute shift:                 -9.5518%



### Report this, don't bury it

Dropping duplicates just moved the headline cancellation rate from **36.04%** down to
**27.49%** — an absolute swing of roughly 8.5 percentage points, and a large relative change.
That is not a rounding effect: it is a direct consequence of a data-cleaning decision we
made.

> **Practical takeaway:** if you hand a stakeholder a "36% of bookings get canceled" number
> and later discover it should have been "27%" after cleaning, that is not a footnote — it
> can change staffing plans, overbooking policy, and revenue forecasts. Whenever a cleaning
> step materially changes a headline metric, say so explicitly, in writing, next to the
> number. Silent cleaning is how "the model said 36%" turns into an argument six months
> later about whose numbers were right.

We are keeping `reservation_status` and `reservation_status_date` in the saved file for now
— those columns are a leakage problem, not a duplication problem, and we handle them
explicitly (with a demonstration of why) in `02_data_processing.ipynb`. We are also leaving
the missing values in `children`, `country`, `agent`, and `company` untouched here, for the
same reason: this notebook's job is deduplication, not imputation.


In [10]:
OUT_PATH = "data/hotel_bookings_clean.csv"
df_clean.to_csv(OUT_PATH, index=False)
print(f"Saved deduplicated dataset to {OUT_PATH}")
print(f"Shape: {df_clean.shape}")

Saved deduplicated dataset to data/hotel_bookings_clean.csv
Shape: (87396, 32)



## 5. Summary

- Loaded 119,390 rows × 32 columns of hotel booking data across two hotels.
- Identified informative missingness in four columns (`children`, `country`, `agent`,
  `company`), each for a different underlying reason — to be handled explicitly in the
  processing notebook.
- Found that **31,994 rows (26.8%)** are fully duplicated across all 32 columns, and that the
  **absence of any primary key** makes it genuinely ambiguous whether these are duplicate log
  entries or coincidentally identical bookings.
- Decided to drop exact duplicates, and explicitly reported the resulting shift in
  cancellation rate (**36.04% → 27.49%**) rather than letting it pass silently.
- Saved the deduplicated (but not yet leakage-cleaned or feature-engineered) dataset to
  `data/hotel_bookings_clean.csv` for use in `01_eda.ipynb` and beyond.
